# Spotify Q&A RAG Pipeline

Inspect the production retrieval and open-source generation pipeline used by FastAPI. No paid API key is required.

In [ ]:
# Run once if dependencies are missing:
# %pip install -r requirements.txt
from pathlib import Path
from spotify_rag.config import Settings
from spotify_rag.assistant import SpotifyAssistant

## Initialize Chroma, multilingual embeddings, and the local LLM

The first execution downloads `paraphrase-multilingual-MiniLM-L12-v2` and `google/flan-t5-small`, then persists the Spotify vector index in `chroma_db/`.

In [ ]:
settings = Settings.from_env(Path.cwd())
assistant = SpotifyAssistant(settings)
assistant.initialize()
print({
    'knowledge_entries': assistant.retriever.count,
    'retrieval_backend': assistant.retriever.backend_name,
    'llm_backend': assistant.generator.backend_name,
    'llm_ready': assistant.generator.ready,
})

## Inspect semantic retrieval

In [ ]:
query = 'How many songs can I save before a flight?'
for item in assistant.retriever.search(query, top_k=4):
    print(f'{item.relevance:.3f}  {item.title}  {item.url}')

## Run the integrated pipeline

The result includes the generated answer, NLP routing metadata, an escalation flag, grounding status, and official source citations.

In [ ]:
result = assistant.chat(query, top_k=4)
result

## Hallucination check

An unrelated question should fall below the relevance threshold and return a safe refusal instead of sending weak context to the LLM.

In [ ]:
assistant.chat('What is the tensile strength of lunar concrete?')